# Análise Exploratória dos Dados (EDA)

**TP Final INF420** — Classificação automática da dificuldade de questões de programação.

Objetivo: carregar a base consolidada (`data/raw/questoes.csv`, gerada pela Etapa 1 — `src.ingest`), entender a **distribuição dos 5 níveis** de dificuldade (muito fácil → muito difícil) e o **tamanho dos enunciados** antes do pré-processamento.

In [ ]:
import sys
from pathlib import Path

# Garante que a raiz do projeto esteja no sys.path para importar 'src'.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data_utils, ingest

# Gera data/raw/questoes.csv a partir de arquivos/ se ainda não existir.
ingest.ensure_dataset()

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
print('Colunas esperadas -> texto:', config.TEXT_COL, '| rotulo:', config.LABEL_COL)
print('Niveis:', config.DIFFICULTY_LABELS)

## 1. Carregar a base

A base vem de `data/raw/questoes.csv`, consolidada pela Etapa 1 a partir da fonte ativa `arquivos/INF110/` (enunciados + `feedbacks_*.json`). As 138 questões aparecem aqui; as 58 sem avaliação ficam com `dificuldade` vazia.

In [ ]:
df = data_utils.load_raw()
print('Dimensoes:', df.shape)
print('Colunas:', list(df.columns))
df.head()

In [ ]:
df.info()

In [ ]:
# Valores ausentes por coluna
df.isna().sum()

## 2. Distribuição dos 5 níveis de dificuldade

Considera apenas as questões **rotuladas** (com avaliação de aluno).

In [ ]:
col_label = config.LABEL_COL
# Apenas questões rotuladas (descarta as 58 sem avaliação).
rotulos = df[col_label].dropna().astype(str).str.strip().str.lower()
print(f'Questoes rotuladas: {len(rotulos)} de {len(df)} (as demais nao tem avaliacao)\n')

contagem = rotulos.value_counts().reindex(config.DIFFICULTY_LABELS).dropna().astype(int)
print(contagem)
print('\nProporcao (%):')
print((contagem / contagem.sum() * 100).round(1))

ax = contagem.plot(kind='bar', color='steelblue')
ax.set_title('Distribuicao dos niveis de dificuldade')
ax.set_xlabel('Dificuldade'); ax.set_ylabel('Qtd. de questoes')
plt.tight_layout(); plt.show()

## 3. Tamanho dos enunciados

Quantidade de caracteres e de palavras por enunciado.

In [ ]:
col_text = config.TEXT_COL
df['n_chars'] = df[col_text].astype(str).str.len()
df['n_palavras'] = df[col_text].astype(str).str.split().map(len)
df[['n_chars', 'n_palavras']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['n_chars'].plot(kind='hist', bins=40, ax=axes[0], color='teal')
axes[0].set_title('Tamanho do enunciado (caracteres)')
df['n_palavras'].plot(kind='hist', bins=40, ax=axes[1], color='darkorange')
axes[1].set_title('Tamanho do enunciado (palavras)')
plt.tight_layout(); plt.show()

## 4. Tamanho do enunciado por nivel de dificuldade

Enunciados mais longos tendem a ser mais dificeis? O boxplot ajuda a ver.

In [ ]:
# Usa só as questões rotuladas, alinhando pelo índice.
y_palavras = df.loc[rotulos.index, 'n_palavras']
ordem = [c for c in config.DIFFICULTY_LABELS if c in set(rotulos)]
ax = sns.boxplot(x=rotulos, y=y_palavras, order=ordem)
ax.set_title('Numero de palavras por nivel de dificuldade')
ax.set_xlabel('Dificuldade'); ax.set_ylabel('Numero de palavras')
plt.tight_layout(); plt.show()

## 5. Exemplos de enunciados por classe

In [ ]:
for nivel in [c for c in config.DIFFICULTY_LABELS if c in set(rotulos)]:
    idx = rotulos[rotulos == nivel].index[0]
    exemplo = df.loc[idx, col_text]
    print(f'\n===== {nivel.upper()} =====')
    print(str(exemplo)[:400], '...')

## 6. Conclusoes da EDA

- Verificar se as classes estao **balanceadas** (senao, usar `class_weight` / metricas macro — ja configurado nos modelos).
- Observar se ha **diferenca de tamanho** dos enunciados entre os niveis.
- Conferir **ruido** no texto (codigo, simbolos) que a limpeza deve remover.

**Proximo passo:** `python -m src.preprocess` (Etapa 2).